## Setup & imports

In [2]:
import os
import gc
import torch
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO, settings
from IPython.display import display

settings.update({"sync": False})
torch.cuda.is_available()

True

In [ ]:
import os
label_path = r"D:\cctv\stage\datasets\People.v3i.yolov8\train\labels"
max_class_id = 0
for file in os.listdir(label_path):
    with open(os.path.join(label_path, file), "r") as f:
        for line in f:
            cls = int(line.split()[0])
            if cls > max_class_id:
                print(f"Error in {file}: Found class {cls}")

## Train & compare YOLO versions

In [ ]:
DATA_YAML = r"D:\cctv\stage\datasets\People.v1i.yolov8\data.yaml"
versions = ["yolov8n", "yolov10n", "yolo11n", "yolo26n"]
results_data = []

for v in versions:
    model = YOLO(v)
    print(f"--- training {v} ---")
    model.train(
        data=DATA_YAML,
        epochs=15,
        imgsz=544,
        device=0,
        batch=16,
        name=f"train_{v}",
        save=True,
        plots=True,
        exist_ok=True,
    )

    metrics = model.val()

    del model
    gc.collect()
    torch.cuda.empty_cache()

    mAP50 = metrics.results_dict.get("metrics/mAP50(B)") or metrics.results_dict.get("metrics/mAP50(BOX)")
    mAP5095 = metrics.results_dict.get("metrics/mAP50-95(B)") or metrics.results_dict.get("metrics/mAP50-95(BOX)")

    results_data.append({
        "Model": v,
        "mAP50": mAP50,
        "mAP50-95": mAP5095,
        "Inference_ms": metrics.speed["inference"],
        "Preprocess_ms": metrics.speed["preprocess"],
        "Postprocess_ms": metrics.speed["postprocess"],
    })

df = pd.DataFrame(results_data)
print("\n--- FINAL COMPARISON ---")
print(df)

mAP_scores = [d["mAP50"] for d in results_data]
inference_times = [d["Inference_ms"] for d in results_data]
colors = ["#3498db", "#2ecc71", "#e74c3c", "#9b59b6"]

fig, ax1 = plt.subplots(figsize=(10, 6))
bars = ax1.bar(versions, mAP_scores, color=colors, alpha=0.6, label="mAP50")
ax1.set_xlabel("YOLO Models")
ax1.set_ylabel("Precision (mAP50)", color="#2980b9", fontsize=12)
ax1.set_ylim(0, 1.1)
ax1.tick_params(axis="y", labelcolor="#2980b9")

ax2 = ax1.twinx()
ax2.plot(versions, inference_times, color="#c0392b", marker="o", linewidth=3, markersize=10, label="Latency (ms)")
ax2.set_ylabel("Inference time (ms)", color="#c0392b", fontsize=12)
ax2.tick_params(axis="y", labelcolor="#c0392b")

plt.title("Performance vs Speed : CCTV Benchmark", fontsize=15, pad=20)
ax1.grid(True, axis="y", linestyle="--", alpha=0.5)
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width() / 2., height + 0.02,
             f"{height:.2f}", ha="center", va="bottom", fontweight="bold")
for i, txt in enumerate(inference_times):
    ax2.annotate(f"{txt:.1f}ms", (versions[i], inference_times[i]),
                 textcoords="offset points", xytext=(0, 10), ha="center", color="#c0392b")
plt.tight_layout()
plt.show()

In [ ]:
MODEL_NAMES = ["yolov8n", "yolov8s", "yolov8m"]
results_data = []

for name in MODEL_NAMES:
    model = YOLO(name)
    print(f"--- training {name} ---")
    model.train(
        data=DATA_YAML,
        epochs=15,
        imgsz=544,
        device=0,
        batch=16,
        name=f"train_{name}_scale",
        save=True,
        plots=True,
        exist_ok=True,
    )

    metrics = model.val()

    del model
    gc.collect()
    torch.cuda.empty_cache()

    mAP50 = metrics.results_dict.get("metrics/mAP50(B)") or metrics.results_dict.get("metrics/mAP50(BOX)")
    mAP5095 = metrics.results_dict.get("metrics/mAP50-95(B)") or metrics.results_dict.get("metrics/mAP50-95(BOX)")

    results_data.append({
        "Model": name,
        "mAP50": mAP50,
        "mAP50-95": mAP5095,
        "Inference_ms": metrics.speed["inference"],
        "Preprocess_ms": metrics.speed["preprocess"],
        "Postprocess_ms": metrics.speed["postprocess"],
    })

df = pd.DataFrame(results_data)
print("\n--- YOLOv8 SCALE COMPARISON ---")
print(df)

mAP_scores = [d["mAP50"] for d in results_data]
inference_times = [d["Inference_ms"] for d in results_data]
colors = ["#3498db", "#2ecc71", "#e74c3c"]

fig, ax1 = plt.subplots(figsize=(10, 6))
bars = ax1.bar(MODEL_NAMES, mAP_scores, color=colors, alpha=0.6, label="mAP50")
ax1.set_xlabel("YOLOv8 Models")
ax1.set_ylabel("Precision (mAP50)", color="#2980b9", fontsize=12)
ax1.set_ylim(0, 1.1)
ax1.tick_params(axis="y", labelcolor="#2980b9")

ax2 = ax1.twinx()
ax2.plot(MODEL_NAMES, inference_times, color="#c0392b", marker="o", linewidth=3, markersize=10, label="Latency (ms)")
ax2.set_ylabel("Inference time (ms)", color="#c0392b", fontsize=12)
ax2.tick_params(axis="y", labelcolor="#c0392b")

plt.title("YOLOv8 Scale Comparison : CCTV Benchmark", fontsize=15, pad=20)
ax1.grid(True, axis="y", linestyle="--", alpha=0.5)
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width() / 2., height + 0.02,
             f"{height:.2f}", ha="center", va="bottom", fontweight="bold")
for i, txt in enumerate(inference_times):
    ax2.annotate(f"{txt:.1f}ms", (MODEL_NAMES[i], inference_times[i]),
                 textcoords="offset points", xytext=(0, 10), ha="center", color="#c0392b")
plt.tight_layout()
plt.show()

In [7]:
import glob
import shutil
import os

EXPORT_DIR = r"D:\cctv\stage\production"
os.makedirs(EXPORT_DIR, exist_ok=True)

best_weights = sorted(glob.glob(r"D:\ECG generation\runs\detect\train_yolov8n_prod\weights\best.pt"))
print(f"Loading trained weights: {best_weights[-1]}")
model = YOLO(best_weights[-1])

print("\n--- Exporting to ONNX for FastAPI + CPU ---")
exported_path = model.export(
    format="onnx",
    imgsz=544,
    half=False,
    dynamic=True,
    opset=17,
    simplify=True,
)

onnx_dst = os.path.join(EXPORT_DIR, "yolov8n_person_det.onnx")
shutil.copy2(exported_path, onnx_dst)
print(f"\nModel exported to: {onnx_dst}")
print(f"\nIn FastAPI, load with:")
print(f"  import onnxruntime")
print(f"  sess = onnxruntime.InferenceSession('{onnx_dst}', providers=['CPUExecutionProvider'])")


Loading trained weights: D:\ECG generation\runs\detect\train_yolov8n_prod\weights\best.pt

--- Exporting to ONNX for FastAPI + CPU ---
Ultralytics 8.4.43  Python-3.12.5 torch-2.5.1+cu121 CPU (12th Gen Intel Core i9-12900K)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'D:\ECG generation\runs\detect\train_yolov8n_prod\weights\best.pt' with input shape (1, 3, 544, 544) BCHW and output shape(s) (1, 5, 6069) (5.9 MB)

ONNX: starting export with onnx 1.21.0 opset 17...
ONNX: slimming with onnxslim 0.1.92...
ONNX: export success  1.5s, saved as 'D:\ECG generation\runs\detect\train_yolov8n_prod\weights\best.onnx' (12.0 MB)

Export complete (1.7s)
Results saved to D:\ECG generation\runs\detect\train_yolov8n_prod\weights
Predict:         yolo predict task=detect model=D:\ECG generation\runs\detect\train_yolov8n_prod\weights\best.onnx imgsz=544 
Validate:        yolo val task=detect model=D:\ECG generation\runs\detect\train_yolov8n_prod\w